In [23]:
import os
import sys
import json

import pandas as pd

In [24]:
project_root = "/home/jcibeira/INA/cell_classifier_yolo" # Modify this path as needed

In [25]:
sys.path.append(str(project_root))
os.chdir(project_root)

In [26]:
df = pd.read_csv("./data/expert_tags/raw/automating-allium-cepa-assay-analysis-with-ai-classifications.csv")

In [27]:
# 1️⃣ Drop all rows that arent from the experts
df_filtered = df[df['user_name'].str.lower() != 'nictauro']

In [28]:
# 2️⃣ Extract filename from subject_data JSON
def extract_filename(subject_json):
    try:
        data = json.loads(subject_json)
        # subject_data is a dict with one key, e.g. {"110828150": {...}}
        first_key = next(iter(data))
        return data[first_key].get("Filename", "").lower()
    except Exception:
        return None

# 3️⃣ Extract class (value) from annotations JSON
def extract_class(annotation_json):
    try:
        data = json.loads(annotation_json)
        # annotation_json is a list of dicts
        return data[0].get("value", "").lower()
    except Exception:
        return None

In [29]:
# 4️⃣ Apply extraction functions
df_filtered["filename"] = df_filtered["subject_data"].apply(extract_filename)
df_filtered["class"] = df_filtered["annotations"].apply(extract_class)

# 5️⃣ Keep only desired columns and normalize case
df_clean = df_filtered[["user_name", "filename", "class"]].copy()
df_clean["user_name"] = df_clean["user_name"].str.lower()

/tmp/ipykernel_26793/3342265411.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered["filename"] = df_filtered["subject_data"].apply(extract_filename)
/tmp/ipykernel_26793/3342265411.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered["class"] = df_filtered["annotations"].apply(extract_class)


In [30]:
df_clean

,user_name,filename,class
7,not-logged-in-893aeb01becde65648fe,a_24_1.png,prophase
8,not-logged-in-893aeb01becde65648fe,a_23_48.png,metaphase
9,not-logged-in-893aeb01becde65648fe,a_7_62.png,interphase
10,not-logged-in-893aeb01becde65648fe,a_15_71.png,interphase
11,not-logged-in-893aeb01becde65648fe,a_23_31.png,prophase
...,...,...,...
2007,alejandrosd,f_13_32.png,indeterminate
2008,alejandrosd,b_90_1.png,interphase
2009,alejandrosd,f_77_34.png,prophase
2010,alejandrosd,a_279_0.png,chromosomal aberrations


In [31]:
df_clean.to_csv("./data/expert_tags/processed/expet_annotations.csv", index=False)